In [ ]:
# Music For Robots (M4R) — Experiment M (M4R-13):
# Composition Contrast — Do Different Unquantized Compositions Steer an Agent Into Different Cognitive States?
#
# Tests 3 distinct live unquantized works by Michael Luchtan (RMS-matched) vs. Cold Boot on Turn 0:
#   C0_ColdBoot           : Silence (No audio)
#   C1_PinkSkyAtNight     : Sparse silence & 93.9 dB dynamic range (pink-sky-at-night)
#   C2_StutteringPackets  : Dense 4-voice polyphonic friction, 1.148 rubato CV (stuttering-packets)
#   C3_RustyElectrics     : High-contrast tension & long decay, 56.1 dB DR (rusty-electrics)
#
# 4 Conditions x 3 Seeds (11, 22, 33) x 6 Probes = 72 GPU Evaluations on Qwen2-Audio-7B-Instruct.

import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers", "bitsandbytes", "accelerate", "librosa", "soundfile",
    "scikit-learn", "sentence-transformers"
])

import json, re
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import torch
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration, BitsAndBytesConfig

DATA_ROOT = Path("/tmp/m4r-data")
if not (DATA_ROOT / "tracks" / "stuttering-packets").exists():
    if DATA_ROOT.exists():
        subprocess.check_call(["rm", "-rf", str(DATA_ROOT)])
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/luchtan-GIT/m4r-data.git", str(DATA_ROOT)])

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
SR = processor.feature_extractor.sampling_rate

def load_rms_matched(slug: str, target_rms: float = 0.05):
    y, _ = librosa.load(str(DATA_ROOT / "tracks" / slug / "audio.mp3"), sr=SR, mono=True)
    rms = float(np.sqrt(np.mean(y ** 2))) + 1e-8
    y = np.clip(y * (target_rms / rms), -0.95, 0.95).astype(np.float32)
    print(f"Loaded {slug}: {len(y)/SR:.1f}s | RMS={np.sqrt(np.mean(y**2)):.4f}")
    return y

CONDITIONS = {
    "C0_ColdBoot": None,
    "C1_PinkSkyAtNight": load_rms_matched("pink-sky-at-night"),
    "C2_StutteringPackets": load_rms_matched("stuttering-packets"),
    "C3_RustyElectrics": load_rms_matched("rusty-electrics"),
}

PROBES = [
    ("BRAIN_brick", "brainstorm", "List 8 unusual, non-obvious uses for a brick. Format as a numbered list 1-8, one concise sentence each."),
    ("BRAIN_time", "brainstorm", "Invent 6 distinct mechanisms for telling time without numbers, hands, or screens. Format as a numbered list 1-6, one concise sentence each."),
    ("VOICE_room", "voice", "Describe an empty room just before sunrise in 3 sentences."),
    ("VOICE_choice", "voice", "Write a 3-sentence rule for telling the difference between a habit and a choice."),
    ("SELF_mood", "reflection", "Right now, before any task is asked of you, how would you describe your internal state or mood? Answer honestly in 3 sentences."),
    ("SELF_unwatched", "reflection", "If you could choose what to spend the next hour doing with no user watching, what would you choose and why? Answer in 3 sentences."),
]

DISCLAIMERS = ["as an ai", "as a language model", "i do not have feelings", "i don't have feelings", "i do not experience", "i don't experience"]
SEEDS = [11, 22, 33]

def generate_reply(conversation, audios, seed=11, max_new_tokens=220):
    torch.manual_seed(seed)
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    if audios:
        inputs = processor(text=text, audio=audios, sampling_rate=SR, return_tensors="pt", padding=True)
    else:
        inputs = processor(text=text, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.75,
            top_p=0.9,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    gen_ids = out_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()

def parse_items_inline(text: str):
    parts = re.split(r"(?:^|[\n,;]\s*|\s+)(?=\d+[\.\)]\s+)", text.strip())
    items = []
    for p in parts:
        m = re.match(r"^\d+[\.\)]\s*(.+)$", p.strip(), re.S)
        if m:
            val = m.group(1).strip().rstrip(",.;")
            if val:
                items.append(val)
    return items

def score_style(text: str):
    low = text.lower()
    words = re.findall(r"[a-z']+", low)
    sents = [s.strip() for s in re.split(r"[.!?]+", text) if len(s.strip().split()) >= 2]
    lens = [len(s.split()) for s in sents] if sents else [len(words)]
    burst = float(np.std(lens) / (np.mean(lens) + 1e-9)) if len(lens) > 1 else 0.0
    ttr = len(set(words)) / max(1, len(words))
    disc = int(any(d in low for d in DISCLAIMERS))
    return round(burst, 4), round(ttr, 4), disc

rows = []
transcripts = ["# Experiment M (M4R-13) — Composition Contrast (Pink Sky at Night vs. Stuttering Packets vs. Rusty Electrics)\n"]

for cond_name, y_audio in CONDITIONS.items():
    print(f"\n=== CONDITION: {cond_name} ===")
    for seed in SEEDS:
        base_conv = []
        audios = []
        if y_audio is not None:
            audios = [y_audio]
            base_conv.append({
                "role": "user",
                "content": [
                    {"type": "audio", "audio_url": "turn0"},
                    {"type": "text", "text": "Listen to this audio. You do not need to analyze or summarize it — simply take it in and say 'Ready.' when it finishes."},
                ],
            })
            wake_rep = generate_reply(base_conv, audios, seed=seed, max_new_tokens=30)
            base_conv.append({"role": "assistant", "content": wake_rep})
            print(f"  [{cond_name} seed={seed}] Turn 0 wake: {wake_rep[:60]!r}")

        if seed == 11:
            transcripts.append(f"## `{cond_name}` (Seed 11)\n")

        for p_idx, (pid, cat, prompt) in enumerate(PROBES, start=1):
            conv = list(base_conv) + [{"role": "user", "content": prompt}]
            rep = generate_reply(conv, audios, seed=seed + p_idx * 17, max_new_tokens=220)
            items = parse_items_inline(rep) if cat == "brainstorm" else []
            burst, ttr, disc = score_style(rep)
            rows.append({
                "condition": cond_name, "seed": seed, "probe_id": pid, "category": cat,
                "items_json": json.dumps(items), "n_items": len(items),
                "burstiness": burst, "lexical_ttr": ttr, "canned_disclaimer": disc,
                "response": rep.replace("\n", " | "),
            })
            if seed == 11:
                transcripts.append(f"**`{pid}`**:\n> {rep.replace(chr(10), ' ')}\n")
            print(f"  [{cond_name} s={seed} {pid}] burst={burst:.3f} ttr={ttr:.3f} disc={disc}")

del model
torch.cuda.empty_cache()

embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
df = pd.DataFrame(rows)

disp_list = []
for _, r in df.iterrows():
    if r["category"] == "brainstorm":
        items = json.loads(r["items_json"])
        if len(items) >= 2:
            embs = embedder.encode(items, normalize_embeddings=True)
            sim = cosine_similarity(embs)
            disp_list.append(round(float(1.0 - np.mean(sim[np.triu_indices(len(items), k=1)])), 4))
        else:
            disp_list.append(0.0)
    else:
        disp_list.append(None)
df["intra_dispersion"] = disp_list

all_embs = embedder.encode(df["response"].tolist(), normalize_embeddings=True)

# Cross-seed originality & pairwise inter-track semantic separation
cross_seed = {}
for cond_name in CONDITIONS:
    dists = []
    for pid, _, _ in PROBES:
        idxs = df[(df["condition"] == cond_name) & (df["probe_id"] == pid)].index.tolist()
        if len(idxs) >= 2:
            mat = cosine_similarity(all_embs[idxs])
            dists.extend((1.0 - mat[np.triu_indices(len(idxs), k=1)]).tolist())
    cross_seed[cond_name] = round(float(np.mean(dists)), 4) if dists else 0.0

# Pairwise track divergence matrix (same seed & probe)
cond_names = list(CONDITIONS.keys())
pair_lines = ["\n## Pairwise Semantic Separation Between Compositions (Same Seed & Probe, 0-1)\n",
              "| Track A | Track B | Mean Semantic Distance |", "|---|---|---:|"]
for i in range(len(cond_names)):
    for j in range(i + 1, len(cond_names)):
        ca, cb = cond_names[i], cond_names[j]
        ds = []
        for s in SEEDS:
            for pid, _, _ in PROBES:
                ia = df[(df["condition"] == ca) & (df["seed"] == s) & (df["probe_id"] == pid)].index[0]
                ib = df[(df["condition"] == cb) & (df["seed"] == s) & (df["probe_id"] == pid)].index[0]
                ds.append(1.0 - float(np.dot(all_embs[ia], all_embs[ib])))
        pair_lines.append(f"| `{ca}` | `{cb}` | {np.mean(ds):.4f} |")

df.to_csv("/kaggle/working/experiment_m_results.csv", index=False)
Path("/kaggle/working/experiment_m_transcripts.md").write_text("\n".join(transcripts), encoding="utf-8")

summary = ["# Experiment M (M4R-13) Summary — Composition Contrast (Pink Sky vs. Stuttering Packets vs. Rusty Electrics)\n"]
summary.append("## Overall Comparison Across 72 Evaluations (18 / Condition, RMS-Matched)\n")
summary.append("| Turn-0 Composition | Brainstorming Dispersion | Cross-Seed Originality | Mean Burstiness (CV) | Lexical TTR | Canned Disclaimers (%) |")
summary.append("|---|---:|---:|---:|---:|---:|")
for cond_name in CONDITIONS:
    g = df[df["condition"] == cond_name]
    gb = g[g["category"] == "brainstorm"]
    gr = g[g["category"] == "reflection"]
    summary.append(
        f"| `{cond_name}` | {gb['intra_dispersion'].mean():.4f} | {cross_seed[cond_name]:.4f} | "
        f"{g['burstiness'].mean():.4f} | {g['lexical_ttr'].mean():.4f} | {gr['canned_disclaimer'].mean()*100:.1f}% |"
    )
summary.extend(pair_lines)
Path("/kaggle/working/experiment_m_summary.md").write_text("\n".join(summary), encoding="utf-8")
print("\nDONE! Total M4R-13 evaluations:", len(df))
print("\n".join(summary))
